# 02 · The Specialist Tools (the deterministic evidence layer)

The specialists in notebook `01` are only as good as the tools they call. Those
tools live in `agent/tools.py` and they are deliberately **deterministic**: same
inputs, same outputs, no generative content in the retrieval/scoring path. The
LLM decides *which* tool to call and *how to present* the result — but the
numbers themselves come from code, not from a model.

This notebook calls those tools **directly, with no agent in the loop**, so you
can see the raw evidence the specialists reason over. Everything here runs
against the local catalog in `data/synthetic-parts-data/` and needs **no**
Bedrock access.

## 0 · Setup

Same bootstrap as notebook `01`: make the repo importable and select the
`synthetic` dataset profile before importing `agent.tools`.

In [1]:
import os, sys, json
from pathlib import Path

# --- Portable data resolution -------------------------------------------------
# This notebook folder is designed to be self-contained: it ships the synthetic
# AnyCompany catalog under notebooks/data/. Prefer that bundled copy so the
# folder works even when lifted into another repo; fall back to the parent
# repo's data/ when running in-place.
NB_DIR = Path.cwd() if (Path.cwd() / "data" / "synthetic-parts-data").exists() else Path.cwd() / "notebooks"
BUNDLED = NB_DIR / "data" / "synthetic-parts-data"
REPO_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
REPO_DATA = REPO_ROOT / "data" / "synthetic-parts-data"

if BUNDLED.exists():
    PARTS_DATA_DIR = BUNDLED
elif REPO_DATA.exists():
    PARTS_DATA_DIR = REPO_DATA
else:
    raise FileNotFoundError("Could not find synthetic-parts-data (bundled or repo).")

# Make PARTS_DATA_DIR authoritative for the imported agent.tools too (it reads
# this env at import time via dataset_profiles), and select the synthetic set.
os.environ["DATASET"] = "synthetic"
os.environ.setdefault("BEDROCK_REGION", "us-east-1")
os.environ["PARTS_DATA_DIR"] = str(PARTS_DATA_DIR)
REGION = os.environ["BEDROCK_REGION"]

# Make the vendored mei_core package importable (this folder is portable;
# the surrounding cells import from mei_core, a self-contained COPY of the
# app's agent + tools that ships in notebooks/mei_core/).
NB_DIR = REPO_ROOT if (REPO_ROOT / "mei_core").exists() else Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

print("parts catalog:", PARTS_DATA_DIR)
print("  (bundled)" if PARTS_DATA_DIR == BUNDLED else "  (repo fallback)")
print("region       :", REGION)


# Optional: if the repo is present, resolve the full profile for KB ids etc.
try:
    from mei_core import dataset_profiles
    print(json.dumps(dataset_profiles.resolve(), indent=2))
except Exception:
    print('(dataset_profiles not importable — running portable/bundled mode)')

parts catalog: /Users/agavili/Desktop/SkyLab-ManufacturingIntelligence/notebooks/data/synthetic-parts-data
  (bundled)
region       : us-east-1


{
  "DATASET": "synthetic",
  "PARTS_DATA_DIR": "/Users/agavili/Desktop/SkyLab-ManufacturingIntelligence/notebooks/data/synthetic-parts-data",
  "PARTS_KB_ID": "KBJSEK4HDV",
  "S3_VECTOR_BUCKET": "skylab-mfg-vectors",
  "S3_VECTOR_INDEX": "reaction-wheels-synthetic"
}


## 1 · The catalog itself

Each part is a folder with a `specs.json` (structured specs + lifecycle), a
`description.txt`, datasheet/ICD PDFs, and images. Here's what one part looks
like — note the structured `vendor` field, which the agent prompts treat as
authoritative (they must never *infer* vendor from the `rw_rl_*` folder ID).

In [2]:
from mei_core.tools import PARTS_DATA_DIR

part_ids = sorted(p.name for p in Path(PARTS_DATA_DIR).iterdir() if p.is_dir())
print('parts in this catalog:', part_ids)

specs = json.loads((Path(PARTS_DATA_DIR) / 'rw_rl_0060' / 'specs.json').read_text())
print(json.dumps(specs, indent=2))

parts in this catalog: ['rw_rl_0003', 'rw_rl_0010', 'rw_rl_0030', 'rw_rl_0060', 'rw_rl_0400', 'rw_rl_1000', 'rw_rl_5000']
{
  "name": "AnyCompany RW3 0.06",
  "category": "reaction_wheel",
  "vendor": "AnyCompany",
  "model": "RW3-0.06",
  "momentum_storage_nms": 0.06,
  "max_torque_mnm": 20,
  "mass_g": 235,
  "voltage_v": "28",
  "voltage_range_v": "7.5 to 36",
  "operating_temp_c": "-30 to +70",
  "interface": "RS-422 or Redundant RS-485",
  "control_mode": "Speed, Torque",
  "power_w": 1.3,
  "dimensions_mm": "77 x 65 x 37.5",
  "vibration_grms": 14.1,
  "radiation_tolerance_krad": 20,
  "qual_standard": "AnyCompany internal (flight heritage qualified)",
  "heritage": "317 units; 930+ cumulative years since 2014",
  "lifecycle": {
    "lifecycle_status": "Active",
    "production_status": "Full production",
    "estimated_years_to_eol": "10+",
    "last_time_buy": "Not announced",
    "typical_lead_time_weeks": 10,
    "lead_time_risk": "Low",
    "forecast_demand_trend": "Stable",

## 2 · `get_part_specs` — structured lookup with name resolution

`get_part_specs` accepts either the catalog folder ID (`rw_rl_0060`) *or* the
human display name (`RW3 0.06`, `AnyCompany RW3 0.06`). The resolver is the
interesting part: reaction-wheel names carry an inconsistent family prefix
(`RW 0.03`, `RW3 0.06`, `RW4 0.4`), so it also matches on the unique **momentum
value** — meaning `RW 0.4`, `RW4 0.4`, and `RW4-0.4` all resolve to the same part.

In [3]:
from mei_core.tools import get_part_specs, _resolve_part_id

for identifier in ['rw_rl_0060', 'RW3 0.06', 'AnyCompany RW3 0.06', 'RW3-0.06']:
    resolved = _resolve_part_id(identifier)
    print(f'{identifier:24s} -> {resolved}')

print()
specs = get_part_specs('RW3 0.06')  # resolves by display name
print('name  :', specs['name'])
print('vendor:', specs['vendor'])
print('momentum_storage_nms:', specs['momentum_storage_nms'])

rw_rl_0060               -> rw_rl_0060
RW3 0.06                 -> rw_rl_0060
AnyCompany RW3 0.06      -> rw_rl_0060
RW3-0.06                 -> rw_rl_0060

name  : AnyCompany RW3 0.06
vendor: AnyCompany
momentum_storage_nms: 0.06


In [4]:
# The momentum-value fallback: a correct value with a wrong/absent family digit
# still resolves, because the value (0.4 Nms) is unique in the catalog.
from mei_core.tools import _extract_momentum_value

for q in ['RW 0.4', 'RW4 0.4', 'RW4-0.4', '0.4 Nms']:
    print(f'{q:12s}  value={_extract_momentum_value(q.lower())}  -> {_resolve_part_id(q)}')

RW 0.4        value=0.4  -> rw_rl_0400
RW4 0.4       value=0.4  -> rw_rl_0400
RW4-0.4       value=0.4  -> rw_rl_0400
0.4 Nms       value=0.4  -> rw_rl_0400


## 3 · CATALOG specialist tools: discovery vs. exhaustive filter

The Catalog specialist has two ways to find parts, and the distinction matters:

- **`kb_search`** — semantic/best-effort retrieval from the Knowledge Base. Good
  for *"find parts similar to..."*. In demo mode (no KB configured) it does a
  keyword match over local JSON.
- **`filter_parts_by_spec`** — an **exhaustive** scan of every part against a
  concrete criterion. Guaranteed complete, never misses — the right tool for
  *"which wheels have at least 0.05 Nms?"*.

In [5]:
from mei_core.tools import filter_parts_by_spec

# Note: value is a string; 'gte' compares it numerically. Results come back
# under 'matches' (each row: part_id, name, field_value).
hits = filter_parts_by_spec(field='momentum_storage_nms', value='0.05', operator='gte')
print(f"{hits['count']} of {hits['total_scanned']} parts have momentum_storage_nms >= 0.05:")
for r in hits['matches']:
    print(f"  {r['part_id']:12s}  {r['name']:22s}  {r['field_value']} Nms")

4 of 7 parts have momentum_storage_nms >= 0.05:
  rw_rl_0060    AnyCompany RW3 0.06     0.06 Nms
  rw_rl_0400    AnyCompany RW4 0.4      0.4 Nms
  rw_rl_1000    AnyCompany RW4 1.0      1.0 Nms
  rw_rl_5000    AnyCompany RW 5.0       5.0 Nms


In [6]:
from mei_core.tools import kb_search

# No KB configured -> demo-mode keyword search over local specs.json files.
res = kb_search(query_text='small low-power reaction wheel', domain='parts', top_k=3)
print(f"{res.get('count')} results (top {min(3, res.get('count', 0))}):")
for r in res.get('results', []):
    print(f"  score={r.get('score'):.2f}  {r.get('item_id')}")

3 results (top 3):
  score=0.96  rw_rl_0003
  score=0.93  rw_rl_0010
  score=0.93  rw_rl_0060


## 4 · EVALUATION specialist tools: deterministic spec-fit

This is the core of the whole app's "evidence, not opinion" stance.
`score_substitution_fit` compares a candidate against a reference (and the
engineer's stated requirements) **spec by spec**, applying the tolerance rules in
`data/requirements_schema.json`, and flags everything out of tolerance. It
returns a `pass`/`flag`/`review`/`unknown` status per spec — and it **never**
recommends or approves.

In [7]:
from mei_core.tools import score_substitution_fit

fit = score_substitution_fit(
    reference_part_id='RW3 0.06',
    candidate_part_id='RW 0.03',
    requirements_json=json.dumps({'momentum_storage_nms': 0.03, 'voltage_v': '5', 'interface': 'ASYNC'}),
)
print('SUMMARY:', fit['summary'])
print('counts :', fit['counts'])
print()
print(f"{'spec':24s} {'status':8s} note")
for r in fit['spec_results']:
    print(f"{r['field']:24s} {r['status']:8s} {r['note']}")
print()
print('DISCLAIMER:', fit['disclaimer'])

SUMMARY: 7 of 8 critical specs pass. 1 flagged, 0 need review, 0 unknown. Not a drop-in replacement; the flagged specs require engineering review.
counts : {'pass': 7, 'flag': 1, 'review': 0, 'unknown': 0}

spec                     status   note
momentum_storage_nms     pass     0.03 >= 0.03
voltage_v                pass     matches ('5' found in '5')
interface                pass     matches ('ASYNC' found in 'ASYNC')
operating_temp_c         pass     [-30.0, 70.0] envelops [-30.0, 70.0]
max_torque_mnm           flag     1.5 < required 20.0
mass_g                   pass     185.0 <= 282 (ref 235.0 +20%)
radiation_tolerance_krad pass     24.0 >= 20.0
qual_standard            pass     same standard (AnyCompany internal (flight heritage qualified))

DISCLAIMER: Evidence only. This tool does not recommend or approve a substitution; the engineer decides.


`compare_parts` is the simpler sibling — a pure field-by-field diff of two parts'
specs, with a match/different verdict per field and no tolerance logic. The
Integration Impact Assessment leans on this for its structured delta.

In [8]:
from mei_core.tools import compare_parts

cmp = compare_parts('rw_rl_0060', 'rw_rl_0030')
print(f"{cmp['part_a_name']}  vs  {cmp['part_b_name']}")
print(f"{cmp['matches']}/{cmp['total_fields']} fields match, {cmp['differences']} differ\n")
for row in cmp['comparison']:
    if row['status'] == 'different':
        print(f"  {row['field']:24s} {row['part_a']}  !=  {row['part_b']}")

AnyCompany RW3 0.06  vs  AnyCompany RW 0.03
5/20 fields match, 15 differ

  dimensions_mm            77 x 65 x 37.5  !=  50 x 50 x 40
  heritage                 317 units; 930+ cumulative years since 2014  !=  90 units; 510+ cumulative years since 2008; common design with RW-0.01
  interface                RS-422 or Redundant RS-485  !=  ASYNC
  lifecycle                {'lifecycle_status': 'Active', 'production_status': 'Full production', 'estimated_years_to_eol': '10+', 'last_time_buy': 'Not announced', 'typical_lead_time_weeks': 10, 'lead_time_risk': 'Low', 'forecast_demand_trend': 'Stable', 'mtbf_hours': None, 'fit_rate': None, 'reliability_prediction_basis': 'Flight heritage (930+ cumulative years, 317 units since 2014); most-flown model', 'weibull_shape_beta': None, 'bathtub_region': 'Useful life (highly mature)', 'recommended_screening': 'Standard flight acceptance'}  !=  {'lifecycle_status': 'Active', 'production_status': 'Full production', 'estimated_years_to_eol': '10+', 'las

## 5 · The requirements schema drives the scoring rules

`score_substitution_fit` isn't hard-coded per part — it reads which specs are
critical, and *how* each is compared, from `data/requirements_schema.json`. That's
what makes the pattern reusable for a different catalog: swap the schema, not the
code. `get_substitution_requirements` exposes it to the agent so it knows what to
ask the engineer for.

In [9]:
from mei_core.tools import get_substitution_requirements

reqs = get_substitution_requirements('reaction_wheel')
print('Specs the engineer must supply:')
for s in reqs['must_ask_user']:
    print(f"  - {s['label']}  (rule: {s['rule']})")
print('\nSpecs compared automatically from the catalog:')
for s in reqs['compared_automatically']:
    tol = f"  +/-{s['tolerance_pct']}%" if s.get('tolerance_pct') else ''
    print(f"  - {s['label']}  (rule: {s['rule']}{tol})")

Specs the engineer must supply:
  - Momentum storage (Nm·s)  (rule: candidate_gte_required)
  - Bus voltage (V)  (rule: match)
  - Control interface  (rule: match)
  - Operating temperature range (°C)  (rule: range_envelops_required)

Specs compared automatically from the catalog:
  - Max torque (mN·m)  (rule: candidate_gte_reference)
  - Mass (g)  (rule: candidate_lte_reference_within_pct  +/-20%)
  - Radiation tolerance (krad)  (rule: candidate_gte_reference)
  - Qualification standard  (rule: qual_review)


## 6 · Compliance: automatic audit logging

Every evaluation is logged for AS9100 traceability (8.4 External Providers /
8.5.6 Control of Changes). In the chat path this is **automatic** — the
orchestrator inspects the Evaluation specialist's tool-call history and writes an
entry per `score_substitution_fit` call, so logging never depends on the model
remembering to do it. Here's the underlying write path.

In [10]:
from mei_core.tools import _write_audit_entry, AUDIT_LOG_PATH

entry = _write_audit_entry('evaluation', json.dumps({
    'reference_id': 'rw_rl_0060', 'candidate_id': 'rw_rl_0030',
    'source': 'notebook_demo',
}))
print('status:', entry['status'])
print('audit log:', AUDIT_LOG_PATH)
print(json.dumps(entry['entry'], indent=2))

status: logged
audit log: /Users/agavili/Desktop/SkyLab-ManufacturingIntelligence/notebooks/data/audit_log.jsonl
{
  "decision_id": "evaluation_20261008_185413_790550",
  "decision_type": "evaluation",
  "timestamp": "2026-10-08T18:54:13.790565+00:00",
  "details": {
    "reference_id": "rw_rl_0060",
    "candidate_id": "rw_rl_0030",
    "source": "notebook_demo"
  },
  "compliance_standard": "AS9100 Rev D",
  "applicable_sections": [
    "8.4 (External Providers)",
    "8.5.6 (Control of Changes)"
  ]
}


## 7 · EVALUATION's visual signal — Nova MME + Claude Vision (inlined)

The Evaluation specialist's `visual_similarity_score` tool is the one place the
system reasons over **pixels**, not specs. It's a two-model signal:

1. **Amazon Nova Multimodal Embeddings** embeds each part's product photo and we
   take the **cosine similarity** — a deterministic, reproducible number.
2. **Claude Vision** looks at both photos and describes concrete physical
   differences (housing, mounting, connector, form factor).

It is a *supplemental* signal, never the basis for a substitution decision.

Below we **reproduce the core snippet inline** (the same code that lives in
`agent/tools.py`) so this notebook is self-contained and portable — it depends
only on `boto3`, `numpy`, and the bundled part photos, not on importing the repo.

_Requires: Bedrock + Nova MME (`amazon.nova-2-multimodal-embeddings-v1:0`) and
Claude Sonnet access._

In [11]:
import base64
import boto3
import numpy as np

NOVA_MME_MODEL_ID = 'amazon.nova-2-multimodal-embeddings-v1:0'
NOVA_MME_DIM = 1024
VISION_MODEL_ID = 'us.anthropic.claude-sonnet-5-5'

bedrock = boto3.client('bedrock-runtime', region_name=REGION)

def embed_image(image_bytes):
    """Embed one image with Nova MME (IMAGE_RETRIEVAL purpose) -> 1024-d vector.
    This is the embedding snippet from agent/tools.py, reproduced inline."""
    b64 = base64.b64encode(image_bytes).decode('utf-8')
    fmt = 'png' if image_bytes[:8] == b'\x89PNG\r\n\x1a\n' else 'jpeg'
    body = {
        'taskType': 'SINGLE_EMBEDDING',
        'singleEmbeddingParams': {
            'embeddingPurpose': 'IMAGE_RETRIEVAL',
            'embeddingDimension': NOVA_MME_DIM,
            'image': {
                'detailLevel': 'STANDARD_IMAGE',
                'format': fmt,
                'source': {'bytes': b64},
            },
        },
    }
    resp = bedrock.invoke_model(
        modelId=NOVA_MME_MODEL_ID, body=json.dumps(body),
        accept='application/json', contentType='application/json',
    )
    result = json.loads(resp['body'].read())
    return np.array(result['embeddings'][0]['embedding'])

print('embed_image() ready — Nova MME', NOVA_MME_MODEL_ID)

embed_image() ready — Nova MME amazon.nova-2-multimodal-embeddings-v1:0


In [12]:
def _photo_path(part_id):
    for ext in ('png', 'jpg', 'jpeg'):
        p = PARTS_DATA_DIR / part_id / f'photo.{ext}'
        if p.exists():
            return p
    return None

def visual_similarity(part_a, part_b):
    """Nova MME cosine similarity + Claude Vision physical-difference analysis
    for two parts' product photos. Portable reproduction of the
    visual_similarity_score tool body in agent/tools.py."""
    pa, pb = _photo_path(part_a), _photo_path(part_b)
    if not pa or not pb:
        return {'error': f'photo missing (a={bool(pa)}, b={bool(pb)})'}
    bytes_a, bytes_b = pa.read_bytes(), pb.read_bytes()

    # --- 1) Nova MME quantitative similarity ---
    emb_a, emb_b = embed_image(bytes_a), embed_image(bytes_b)
    similarity = float(np.dot(emb_a, emb_b) /
                       (np.linalg.norm(emb_a) * np.linalg.norm(emb_b)))

    # --- 2) Claude Vision qualitative differences ---
    fmt_a = 'png' if bytes_a[:8] == b'\x89PNG\r\n\x1a\n' else 'jpeg'
    fmt_b = 'png' if bytes_b[:8] == b'\x89PNG\r\n\x1a\n' else 'jpeg'
    vision_prompt = (
        'Compare these two aerospace reaction wheels. For each, note concrete\n'
        'physical differences: housing size/shape, mounting interface, connector\n'
        'location/type, overall form factor. 4-6 specific bullet points; only\n'
        'describe what you can SEE.'
    )
    analysis = None
    try:
        vr = bedrock.converse(
            modelId=VISION_MODEL_ID,
            messages=[{'role': 'user', 'content': [
                {'image': {'format': fmt_a, 'source': {'bytes': bytes_a}}},
                {'image': {'format': fmt_b, 'source': {'bytes': bytes_b}}},
                {'text': vision_prompt},
            ]}],
            inferenceConfig={'maxTokens': 768},
        )
        # Claude may emit a reasoning block first — scan for the text block.
        blocks = vr['output']['message'].get('content', [])
        analysis = next((b['text'] for b in blocks if isinstance(b, dict) and 'text' in b), None)
    except Exception as e:
        analysis = f'Vision analysis unavailable: {type(e).__name__}: {e}'

    return {
        'part_a': part_a, 'part_b': part_b,
        'visual_similarity': round(similarity, 4),
        'interpretation': ('Very similar form factor' if similarity > 0.85
                           else 'Moderately similar' if similarity > 0.7
                           else 'Visually different form factors'),
        'physical_analysis': analysis,
        'models_used': [NOVA_MME_MODEL_ID, VISION_MODEL_ID],
    }

res = visual_similarity('rw_rl_0060', 'rw_rl_5000')  # RW3 0.06 vs RW 5.0
print('Nova MME cosine similarity:', res.get('visual_similarity'),
      '->', res.get('interpretation'))
print()
print(res.get('physical_analysis', '')[:700])

Nova MME cosine similarity: 0.5116 -> Visually different form factors

Note: I'm describing only what's visible. Neither image has a scale reference, so all size comparisons are relative.

**Image 1: square-frame unit**
- **Housing shape:** A low, square black frame with rounded corners. The top plate is flat, and the side walls have large rectangular cutouts that expose the interior and the base plate.
- **Mounting interface:** A through-hole sits in each of the four corners of the square flange. More small holes are visible on the base plate inside the frame. The unit would bolt down by its corners.
- **Rotor/hub:** A round, machined, dark-finished disc sits in the center of the frame. It has four small socket-head screws and several smaller unfilled holes in


That number and that analysis are exactly what the Evaluation specialist surfaces
as *"Supplemental: Visual Analysis"* in a one-to-one comparison. Because the
embedding is deterministic, the cosine score is reproducible run to run; the
Claude Vision prose varies but stays grounded in the two photos.

In the running app this is the `visual_similarity_score` **tool** (`@tool` in
`agent/tools.py`); notebook `04` §6 calls that tool directly. Here you've seen the
body it runs.

## Recap

- The retrieval/scoring layer is **deterministic** — the agents reason over hard
  evidence, they don't invent specs.
- `filter_parts_by_spec` (exhaustive) vs. `kb_search` (semantic) is a deliberate
  split: completeness when you have a criterion, recall when you have a description.
- `score_substitution_fit` applies tolerance rules from a swappable schema and
  emits per-spec `pass`/`flag`/`review` — never a recommendation.
- Audit logging is automatic and tied to the actual tool calls, not model prose.

**Next:** [`03_streaming_and_gateway.ipynb`](03_streaming_and_gateway.ipynb) shows
how the graph is streamed to the UI, how AgentCore Gateway routes KB + web
search, and how the Integration Impact Assessment differs from this planned graph.